<a href="https://colab.research.google.com/github/PYMaksim/From-220-to-180-Fixing-LLM-Math/blob/main/ai_math_benchmark_yandex_vs_gigachat.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
# -*- coding: utf-8 -*-
"""
Project: YandexGPT Multilingual Reasoning Test
Description: Testing Zero-Shot vs Chain of Thought (CoT) in Russian and English.
"""

from google.colab import userdata
from openai import OpenAI
import time

# --- CONFIGURATION ---
API_KEY = userdata.get('Y')
FOLDER_ID = "b1g2hgrbnnrqll94nr9k"
BASE_URL = "https://ai.api.cloud.yandex.net/v1"
HEADERS = {"Authorization": f"Api-Key {API_KEY}", "x-folder-id": FOLDER_ID}
client = OpenAI(api_key=API_KEY, base_url=BASE_URL)

MODEL_ID = "yandexgpt-5-pro"
MODEL_URI = f"gpt://{FOLDER_ID}/{MODEL_ID}/latest"

# --- UNIVERSAL TASK ---
TASK_RU = "У Маши было 500 рублей. Она купила 3 тетради по 80 рублей и 2 ручки по 40 рублей. Сколько сдачи ей дали?"
TASK_EN = """Masha had 500 rubles. She bought 3 notebooks for 80 rubles each and 2 pens for 40 rubles each. How much change did she get?"""

# --- TEST MATRIX ---
# Мы объединяем языки и техники в один список для цикла.
test_cases = [
    {
        "label": "ZERO-SHOT (RU)",
        "system": "Ты — математик. Отвечай только числом.",
        "user": TASK_RU
    },
    {
        "label": "ZERO-SHOT (EN)",
        "system": "You are a mathematician. Answer with a number only.",
        "user": TASK_EN
    },
    {
        "label": "CHAIN OF THOUGHT (RU)",
        "system": "Ты — математик. Реши задачу пошагово.",
        "user": TASK_RU + "\n\nРеши задачу пошагово. Сначала посчитай стоимость тетрадей, потом ручек, затем вычти из 500."
    },
    {
        "label": "CHAIN OF THOUGHT (EN)",
        "system": "You are a mathematician. Solve step-by-step.",
        "user": TASK_EN + "\n\nLet's think step by step. Show your calculations."
    }
]

# --- EXECUTION ---
print("🚀 Starting Multilingual Reasoning Test...")
results = []

for case in test_cases:
    print(f"\n{'='*20} {case['label']} {'='*20}")
    start = time.time()

    try:
        response = client.chat.completions.create(
            model=MODEL_URI,
            messages=[
                {"role": "system", "content": case["system"]},
                {"role": "user", "content": case["user"]}
            ],
            temperature=0.0,
            max_tokens=400,
            extra_headers=HEADERS
        )

        latency = time.time() - start
        answer = response.choices[0].message.content.strip()

        print(f"✅ SUCCESS | Latency: {latency:.2f}s")
        print("-" * 10 + " ANSWER " + "-" * 10)
        print(answer)

        results.append({
            "test": case["label"],
            "answer": answer,
            "latency": latency
        })

    except Exception as e:
        print(f"❌ ERROR: {e}")
        results.append({"test": case["label"], "status": "ERROR"})

# --- SUMMARY REPORT ---
print("\n\n" + "="*50)
print("📊 FINAL COMPARISON REPORT")
print("="*50)
for r in results:
    if "status" in r:
        print(f"❌ {r['test']}: FAILED")
    else:
        print(f"✅ {r['test']}: {r['answer']} | Time: {r['latency']:.2f}s")

🚀 Starting Multilingual Reasoning Test...

==================== ZERO-SHOT (RU) ====================
✅ SUCCESS | Latency: 0.68s
---------- ANSWER ----------
220

==================== ZERO-SHOT (EN) ====================
✅ SUCCESS | Latency: 0.41s
---------- ANSWER ----------
220

==================== CHAIN OF THOUGHT (RU) ====================
✅ SUCCESS | Latency: 2.36s
---------- ANSWER ----------
Сначала посчитаем стоимость тетрадей:
\[ 3 \times 80 = 240 \text{ рублей} \]

Затем посчитаем стоимость ручек:
\[ 2 \times 40 = 80 \text{ рублей} \]

Теперь сложим стоимость тетрадей и ручек:
\[ 240 + 80 = 320 \text{ рублей} \]

Вычтем общую стоимость покупок из суммы, которая была у Маши:
\[ 500 - 320 = 180 \text{ рублей} \]

Ответ: 180 рублей

==================== CHAIN OF THOUGHT (EN) ====================
✅ SUCCESS | Latency: 3.97s
---------- ANSWER ----------
First, calculate the total cost of the notebooks. Masha bought 3 notebooks, each costing 80 rubles. The total cost for the notebooks 

In [39]:
import requests
import time
import json

# --- CONFIGURATION ---
ACCESS_TOKEN = "eyJ..."


# URL для GigaChat API
GIGACHAT_URL = "https://api.giga.chat/v1/chat/completions"

HEADERS = {
    "Authorization": f"Bearer {ACCESS_TOKEN}",
    "Content-Type": "application/json"
}

MODEL_ID = "GigaChat-2-Pro"

# --- PROMPTS ---
USER_QUESTION_RU = """У Маши было 500 рублей. Она купила 3 тетради по 80 рублей и 2 ручки по 40 рублей. Сколько сдачи ей дали?"""
USER_QUESTION_EN = """Masha had 500 rubles. She bought 3 notebooks for 80 rubles each and 2 pens for 40 rubles each. How much change did she get?"""

SYSTEM_PROMPT = """You are a math tutor. Solve the problem step-by-step."""

# --- EXECUTION ---
print("\n🚀 Testing GigaChat API (Direct - FIXED)...")
results = []

test_cases = [
    {"label": "ZERO-SHOT (RU)", "system": "Ты — математик. Ответь только числом.", "user": USER_QUESTION_RU + "\n\nОтветь только числом."},
    {"label": "ZERO-SHOT (EN)", "system": "You are a mathematician. Answer with a number only.", "user": USER_QUESTION_EN + "\n\nAnswer with a number only."},
    {"label": "CHAIN OF THOUGHT (RU)", "system": "Ты — математик. Реши задачу пошагово.", "user": USER_QUESTION_RU + "\n\nРеши задачу пошагово. Сначала посчитай стоимость тетрадей, потом ручек, затем вычти из 500."},
    {"label": "CHAIN OF THOUGHT (EN)", "system": "You are a mathematician. Solve step-by-step.", "user": USER_QUESTION_EN + "\n\nLet's think step by step. Show your calculations."}
]

for case in test_cases:
    print(f"\n{'='*20} {case['label']} {'='*20}")
    start = time.time()

    payload = {
        "model": MODEL_ID,
        "messages": [
            {"role": "system", "content": case["system"]},
            {"role": "user", "content": case["user"]}
        ],
        "temperature": 0.0,
        "max_tokens": 400
    }

    try:

        response = requests.post(GIGACHAT_URL, headers=HEADERS, json=payload, verify=False)
        response.raise_for_status()

        latency = time.time() - start
        answer = response.json()["choices"][0]["message"]["content"].strip()

        print(f"[SUCCESS] Latency: {latency:.2f}s")
        print("-" * 10 + " ANSWER " + "-" * 10)
        print(answer)

        results.append({"test": case["label"], "answer": answer, "latency": latency})

    except Exception as e:
        print(f"[ERROR] {e}")
        results.append({"test": case["label"], "status": "ERROR"})

# --- SUMMARY REPORT ---
print("\n\n" + "="*50)
print("📊 FINAL COMPARISON REPORT")
print("="*50)
for r in results:
    if "status" in r and r["status"] == "ERROR":
        print(f"❌ {r['test']}: FAILED")
    else:
        print(f"✅ {r['test']}: {r['answer']} | Time: {r['latency']:.2f}s")


🚀 Testing GigaChat API (Direct - FIXED)...

==================== ZERO-SHOT (RU) ====================


/usr/local/lib/python3.13/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.giga.chat'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


[SUCCESS] Latency: 0.98s
---------- ANSWER ----------
140

==================== ZERO-SHOT (EN) ====================


/usr/local/lib/python3.13/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.giga.chat'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


[SUCCESS] Latency: 0.61s
---------- ANSWER ----------
220

==================== CHAIN OF THOUGHT (RU) ====================


/usr/local/lib/python3.13/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.giga.chat'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


[SUCCESS] Latency: 4.10s
---------- ANSWER ----------
### Шаг 1: Найдем стоимость тетрадей
Стоимость одной тетради составляет 80 рублей. Маша купила 3 тетради.

$$
\text{Стоимость тетрадей} = 3 \times 80 = 240 \text{ рублей}
$$

### Шаг 2: Найдем стоимость ручек
Стоимость одной ручки составляет 40 рублей. Маша купила 2 ручки.

$$
\text{Стоимость ручек} = 2 \times 40 = 80 \text{ рублей}
$$

### Шаг 3: Вычислим общую сумму покупки
Теперь сложим стоимости тетрадей и ручек, чтобы найти общую сумму покупки.

$$
\text{Общая сумма покупки} = 240 + 80 = 320 \text{ рублей}
$$

### Шаг 4: Найдем сдачу
Изначально у Маши было 500 рублей. Вычтем из этой суммы общую стоимость покупок, чтобы найти сдачу.

$$
\text{Сдача} = 500 - 320 = 180 \text{ рублей}
$$

==================== CHAIN OF THOUGHT (EN) ====================


/usr/local/lib/python3.13/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.giga.chat'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


[SUCCESS] Latency: 3.51s
---------- ANSWER ----------
Step 1: Calculate the total cost of the notebooks.
- Masha buys 3 notebooks at 80 rubles each.
- Total cost of notebooks = $3 \times 80 = 240$ rubles.

Step 2: Calculate the total cost of the pens.
- Masha buys 2 pens at 40 rubles each.
- Total cost of pens = $2 \times 40 = 80$ rubles.

Step 3: Find the total amount spent on both items.
- Total amount spent = $240 + 80 = 320$ rubles.

Step 4: Determine how much money Masha has left after her purchases.
- Initial amount = 500 rubles.
- Change received = $500 - 320 = 180$ rubles.

Final Answer:
Masha got 180 rubles in change.


📊 FINAL COMPARISON REPORT
✅ ZERO-SHOT (RU): 140 | Time: 0.98s
✅ ZERO-SHOT (EN): 220 | Time: 0.61s
✅ CHAIN OF THOUGHT (RU): ### Шаг 1: Найдем стоимость тетрадей
Стоимость одной тетради составляет 80 рублей. Маша купила 3 тетради.

$$
\text{Стоимость тетрадей} = 3 \times 80 = 240 \text{ рублей}
$$

### Шаг 2: Найдем стоимость ручек
Стоимость одной ручки составляе